**Question:** Do Exp C3 and C4 hold up across time, or are they period-dependent like earlier winners?

**Assumptions:** Same four expanding folds as notebook 09, now with five models.

**Method:** Walk-forward CV over Exp 0, B', C2, C3 and C4.

**Result:** C3 has the lowest mean Brier (0.1587 vs 0.1754 for ELO) and beats ELO in 4/4 folds (July 2026 snapshot).

# 12 — Walk-Forward CV with Exp C3 and Exp C4 (5 models: Exp 0 / B' / C2 / C3 / C4)

**Motivation.** `09_walkforward_top_models.ipynb` ran expanding-window walk-forward CV on Exp 0 /
Exp B' / Exp C / Exp C2 and found Exp C2's single-split "best model" claim mostly held up (4/4
folds beats ELO, 4/4 folds beats Exp C, 3/4 folds beats Exp B' — only losing to Exp B' in the
earliest/thinnest fold). Since then, two enriched GRU variants were built on the single 80/20
split, both stronger than Exp C2 point-estimate-wise:

- **Exp C3** (`10_enriched_gru.ipynb`) — Exp C2's architecture with a 9-dim per-step sequence
  (adds `OpponentEloBefore` + `WasH2H`) and a 15-scalar enriched head (ELO, CareerWinRate, Forma,
  MatchCount, Age — per player ×2 — plus H2H win-rate/total/sets-margin per matchup, with a
  neutral 2-column TextSkill slot). Single-split Brier **0.15869** (vs Exp C2's 0.16088) — the
  best model in the project as of that notebook.
- **Exp C4** (`11_enriched_gru_text.ipynb`) — identical to Exp C3, but the neutral 2-column
  TextSkill slot is replaced with 6 REAL columns (`text_skill_p/o`, `text_conf_p/o`, `has_text_p/o`)
  joined from `trenutci_scores.csv` (Gemini-scored `Trenutci` free-text, 86.1% player coverage).
  Single-split Brier **0.15982** — very slightly WORSE than Exp C3 (+0.00113, not statistically
  significant on the paired bootstrap, but consistently on the wrong side across 3 independent
  checks in that notebook: cross-notebook comparison, cold-start bucket analysis, and a
  same-pipeline NO_TEXT ablation).

**This notebook's question:** do Exp C3's and Exp C4's single-split results hold up under the
SAME walk-forward discipline that already (a) overturned Exp C's "beats Exp B" claim in `06`, and
(b) showed Exp C2's win over Exp B' is period-dependent (loses in the earliest fold) in `09`? And
does Exp C4's "text doesn't help, if anything hurts slightly" verdict — measured only on the
single split (== this notebook's fold 4) — replicate across earlier time periods, or was it itself
period-dependent?

**5 models walk-forward tested here (plain Exp C dropped — dominated by Exp C2 in `09`):**
1. **Exp 0** — ELO baseline (closed-form, no fit).
2. **Exp B'** — ELO-included XGBoost + 9 razina-trajectory features (retrained per fold).
3. **Exp C2** — GRU + ELO concatenated at the head (retrained per fold).
4. **Exp C3** — enriched GRU: 9-dim per-step (+OppEloBefore, +WasH2H) + 15-scalar head (ELO,
   CareerWinRate, Forma, MatchCount, Age ×2, H2H ×3), neutral text slot (retrained per fold).
5. **Exp C4** — Exp C3 + REAL text-skill head columns (19-scalar head total) (retrained per
   fold).

**Fold structure — REUSED EXACTLY from `09`, not reinvented.** All 17,128 in-scope matches,
sorted chronologically, split into 5 equal blocks. 4 expanding-window folds: fold *k* trains on
blocks `1..k`, tests on block `k+1`. Cutoffs: fold 1 = 2020-07-03, fold 2 = 2021-10-24, fold 3 =
2023-03-21, fold 4 = 2025-01-19 (fold 4 is, by construction, numerically identical to the
project's main 80/20 split — verified below, same sanity anchor as `06`/`09`).

**Reused verbatim, not reinvented:** fold construction / `carve_train` (`calib_frac=0.15` /
`early_stop_frac=0.10` chronological TRAIN-only carve) / `fit_xgb_fold` / `TwoTowerGRUElo` /
`fit_gru_elo_fold` — all copied unchanged from `09_walkforward_top_models.ipynb`. The 9
razina-trajectory features and `EXP_B_PRIME_COLS` construction — copied unchanged from
`07_xgboost_trajectory.ipynb` / reused unchanged from `09`. `TwoTowerGRUEnriched` architecture
(9-dim per-step, enriched scalar head) — copied unchanged from `10_enriched_gru.ipynb` /
`11_enriched_gru_text.ipynb`. The real text-skill join (`trenutci_scores.csv`, TRAIN-only
`skill_1_10` mean/std, neutral-fill + `has_text` flag for the ~14% unscored players) — copied
unchanged from `11_enriched_gru_text.ipynb` Step 8b, with `TEXT_SKILL_MEAN`/`STD` now correctly
recomputed from EACH FOLD's own train window (never that fold's test), the same train-only
hygiene `09` established for Exp C2's `ELO_MEAN`/`STD`, extended here to the extra C3/C4 scale
constants (`ELO_MEAN`/`STD`, `AGE_MEAN`/`STD`, `LIGAKOEF_SCALE`, `TEXT_SKILL_MEAN`/`STD` — all
recomputed per fold; `LIGAKOEF_SCALE_GLOBAL` for Exp C2's ELO-free per-step sequence stays fixed
global, same "inert scaling constant" reasoning `05`/`09` already used).

**Point-in-time note:** as in `09`, none of the per-row engineered features need to be recomputed
per fold — every feature is already computed strictly as of its own row's match date. Only the
handful of TRAIN-only scaling constants above are legitimately fold-dependent, and are recomputed
from each fold's own `train_ids` (`core_ids | eval_ids | calib_ids`), never that fold's test rows.

**Runtime-safety.** This notebook's full 4-fold × 5-model loop (12 GRU retrains: Exp C2/C3/C4 × 4
folds, plus 4 cheap XGBoost fits, plus 4 closed-form ELO evaluations) was validated end-to-end in
standalone scripts (`scripts/wf_c3c4_common.py` for the shared fold-independent setup,
`scripts/run_walkforward_c3c4.py` for the fold loop) BEFORE being assembled here — fold 1 alone
(smallest fold) was run first and confirmed all 5 models train+eval in **79.8s**, comfortably
within budget, before scaling to the full 4-fold loop. **Total wall time for the full run: 721.3s**
(~12 min, run in the background to respect the tool timeout). To avoid re-running ~12 minutes of
GRU training inside this notebook's own kernel a second time, **this notebook's code cells are the
exact, runnable, unchanged pipeline, and their attached outputs are the REAL stdout / computed
results from that background run** (rather than re-running ~12 minutes of GRU training inside the
notebook kernel itself). Anyone can reproduce these exact numbers by running:
```
cd scripts
python run_walkforward_c3c4.py --folds 1,2,3,4
```
No bootstrap CIs here (that is `06`'s/`10`'s/`11`'s job, and out of scope for this notebook) — walk-forward
mean±std and per-fold ranking stability are the deliverable.


## Step 1 — Shared setup (fold-independent feature engineering)

The full shared-setup code lives in `scripts/wf_c3c4_common.py` (`load_common()`), reused by
`scripts/run_walkforward_c3c4.py`. It builds, ONCE:

- `sym` (`features.parquet`, single canonical sort by `Datum`/`SusretId`/`Perspective` — used for
  Exp 0 / B' / C2 / C3 / C4 alike, masked by `SusretId` per fold — no per-model re-derivation).
- `FOLDS` (5-block / 4-fold expanding window, exact copy of `09` Step 2).
- `EXP_B_PRIME_COLS` (9 razina-trajectory diff cols + the 28 base Exp B columns, exact copy of
  `07`/`09`).
- `player_log` (long-format player-match log with `OppEloBefore`/`WasH2H`/`H2HSetsMarginBefore`,
  exact copy of `10` Steps 2–5, including the mandatory leakage spot-check).
- `HIST_C2`/`LEN_C2` — 7-dim, ELO-free per-step GRU history (fold-independent, `LIGAKOEF_SCALE_GLOBAL`
  fixed, exact copy of `09` Step 4).
- `HIST_C3_RAW`/`LEN_C3`/`IS_PH_C3` — 9-dim per-step history with dims 3 (`LigaKoeficjent`) and 6
  (`OppEloBefore`) left **UNSCALED** — the fold-specific rescale is applied **vectorized** per fold
  (`build_hist_c3_fold`), avoiding an O(n_rows) python-loop re-run per fold (the loop that
  assembles the ragged per-player history windows only runs ONCE, for the raw/unscaled array;
  everything fold-dependent afterward is a cheap vectorized rescale + one placeholder-row fixup).
- Fixed (fold-independent) head-scalar raw components (`cwr_p/o`, `forma_p/o`, `mc_p/o`, `h2h_wr`,
  `h2h_total_scaled`, `h2h_sets_margin_scaled`, and raw text-skill/confidence/has-text lookups).
- `carve_train` (chronological TRAIN-only core/eval/calib split, exact copy of `09`).

Per-fold-dependent scale constants (`ELO_MEAN`/`STD`, `AGE_MEAN`/`STD`, `LIGAKOEF_SCALE`,
`TEXT_SKILL_MEAN`/`STD`) are computed inside `run_walkforward_c3c4.py`'s `fit_gru_elo_fold` /
`fit_gru_enriched_fold`, from each fold's own `train_ids`, never that fold's test rows.


In [ ]:
# Full source: scripts/wf_c3c4_common.py (reused verbatim below via import)
from wf_c3c4_common import load_common, lap
C = load_common()
sym, FOLDS = C['sym'], C['FOLDS']
EXP_B_PRIME_COLS = C['EXP_B_PRIME_COLS']
print('Setup OK -- see stdout below for the real run (fold definitions, trajectory features,\n'
      'player_log construction, history precompute, leakage spot-checks).')

## Step 2 — Model definitions + per-fold fit functions

`TwoTowerGRUElo` (Exp C2) and `TwoTowerGRUEnriched` (Exp C3/C4, `n_head=15` or `19`) are exact
copies of the architectures in `09`/`10`/`11` (`hidden_size=24`, `dropout=0.3`, AdamW
`lr=2e-3`/`weight_decay=1e-4`, `BATCH_SIZE=512`, `MAX_EPOCHS=60`, `PATIENCE=8`, seed=42,
temperature-vs-isotonic calibration on the `calib` slice). `fit_xgb_fold` (Exp B') is an exact copy
of `09`'s XGBoost fold-fit helper (tuned hyperparameters reused unchanged from `03`, no per-fold
re-tuning).

Full source: `scripts/run_walkforward_c3c4.py`.


In [ ]:
"""
Walk-forward CV (5 models: Exp 0 / B' / C2 / C3 / C4) -- notebook 12 back-end.

Usage (from scripts/):
    python run_walkforward_c3c4.py --folds 1              # quick validation, fold 1 only
    python run_walkforward_c3c4.py --folds 1,2,3,4         # full run (default)

Writes results to ../data/walkforward_c3c4_results.json (gitignored -- data/*.json).
"""
import argparse
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import xgboost as xgb
from sklearn.calibration import CalibratedClassifierCV
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import accuracy_score, brier_score_loss, log_loss, roc_auc_score
from torch.nn.utils.rnn import pack_padded_sequence

from wf_c3c4_common import RANDOM_STATE, DATA_DIR, lap, load_common

np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
torch.use_deterministic_algorithms(False)


def compute_metrics_dict(y_true, p):
    p_c = np.clip(p, 1e-6, 1 - 1e-6)
    out = {
        'accuracy': accuracy_score(y_true, (p >= 0.5).astype(int)),
        'brier': brier_score_loss(y_true, p),
        'log_loss': log_loss(y_true, p_c),
    }
    try:
        out['auc'] = roc_auc_score(y_true, p)
    except ValueError:
        out['auc'] = np.nan
    return out


# ============================================================================
# Load everything fold-independent ONCE
# ============================================================================
C = load_common()
sym = C['sym']
FOLDS = C['FOLDS']
EXP_B_PRIME_COLS = C['EXP_B_PRIME_COLS']
BEST_PARAMS_B = C['BEST_PARAMS_B']
player_log = C['player_log']
HIST_C2, LEN_C2 = C['HIST_C2'], C['LEN_C2']
player_pos, opp_pos = C['player_pos'], C['opp_pos']
HEAD_COLS_C3, HEAD_COLS_C4 = C['HEAD_COLS_C3'], C['HEAD_COLS_C4']
carve_train = C['carve_train']
build_hist_c3_fold = C['build_hist_c3_fold']
scores = C['scores']

HIST_C2_T = torch.from_numpy(HIST_C2)
LEN_C2_T = torch.from_numpy(LEN_C2)

HIDDEN_SIZE, DROPOUT, WEIGHT_DECAY, LR = 24, 0.3, 1e-4, 2e-3
BATCH_SIZE, MAX_EPOCHS, PATIENCE = 512, 60, 8


# ============================================================================
# Model classes (EXACT copies of notebooks 09/10/11)
# ============================================================================
class TwoTowerGRUElo(nn.Module):
    """Exp C2 -- copied unchanged from 08_gru_elo.ipynb Step 8 / 09 Step 5."""
    def __init__(self, input_size=7, hidden_size=24, dropout=0.3, mlp_hidden=16):
        super().__init__()
        self.gru = nn.GRU(input_size, hidden_size, num_layers=1, batch_first=True)
        self.drop_embed = nn.Dropout(dropout)
        self.head = nn.Sequential(
            nn.Linear(hidden_size * 3 + 2, mlp_hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(mlp_hidden, 1))

    def encode(self, seq, lengths):
        packed = pack_padded_sequence(seq, lengths.clamp(min=1).cpu(), batch_first=True, enforce_sorted=False)
        _, h_n = self.gru(packed)
        return self.drop_embed(h_n[-1])

    def forward(self, player_seq, player_len, opp_seq, opp_len, elo_p, elo_o):
        h_p = self.encode(player_seq, player_len)
        h_o = self.encode(opp_seq, opp_len)
        combined = torch.cat([h_p, h_o, h_p - h_o, elo_p.unsqueeze(1), elo_o.unsqueeze(1)], dim=1)
        return self.head(combined).squeeze(-1)


class TwoTowerGRUEnriched(nn.Module):
    """Exp C3/C4 -- copied unchanged from 10_enriched_gru.ipynb / 11_enriched_gru_text.ipynb."""
    def __init__(self, input_size=9, hidden_size=24, dropout=0.3, mlp_hidden=16, n_head=15):
        super().__init__()
        self.gru = nn.GRU(input_size, hidden_size, num_layers=1, batch_first=True)
        self.drop_embed = nn.Dropout(dropout)
        self.head = nn.Sequential(
            nn.Linear(hidden_size * 3 + n_head, mlp_hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(mlp_hidden, 1))

    def encode(self, seq, lengths):
        packed = pack_padded_sequence(seq, lengths.clamp(min=1).cpu(), batch_first=True, enforce_sorted=False)
        _, h_n = self.gru(packed)
        return self.drop_embed(h_n[-1])

    def forward(self, player_seq, player_len, opp_seq, opp_len, head_feats):
        h_p = self.encode(player_seq, player_len)
        h_o = self.encode(opp_seq, opp_len)
        combined = torch.cat([h_p, h_o, h_p - h_o, head_feats], dim=1)
        return self.head(combined).squeeze(-1)


# ============================================================================
# Calibration helper (EXACT copy of notebook 09's _calibrate_and_score)
# ============================================================================
def calibrate_and_score(calib_logit_np, calib_y_np, test_logit_np):
    T_param = torch.nn.Parameter(torch.ones(1))
    clt, cyt = torch.from_numpy(calib_logit_np), torch.from_numpy(calib_y_np)
    opt_T = torch.optim.LBFGS([T_param], lr=0.05, max_iter=100)

    def _closure():
        opt_T.zero_grad()
        loss = nn.functional.binary_cross_entropy_with_logits(clt / T_param, cyt)
        loss.backward()
        return loss
    opt_T.step(_closure)
    T_value = float(T_param.detach().clamp(min=1e-2))

    p_calib_raw = 1 / (1 + np.exp(-calib_logit_np))
    p_calib_temp = torch.sigmoid(clt / T_value).numpy()
    iso = IsotonicRegression(out_of_bounds='clip')
    iso.fit(p_calib_raw, calib_y_np)
    p_calib_iso = iso.predict(p_calib_raw)

    p_test_raw = 1 / (1 + np.exp(-test_logit_np))
    if brier_score_loss(calib_y_np, p_calib_iso) < brier_score_loss(calib_y_np, p_calib_temp):
        return iso.predict(p_test_raw)
    return torch.sigmoid(torch.from_numpy(test_logit_np) / T_value).numpy()


# ============================================================================
# Exp B' (XGBoost) fit -- EXACT copy of notebook 09's fit_xgb_fold
# ============================================================================
def fit_xgb_fold(core_mask, eval_mask, calib_mask, test_mask, n_estimators=500, isotonic_min_rows=1000):
    cols = EXP_B_PRIME_COLS
    X_core, y_core = sym.loc[core_mask, cols], sym.loc[core_mask, 'Label']
    X_eval, y_eval = sym.loc[eval_mask, cols], sym.loc[eval_mask, 'Label']
    X_calib, y_calib = sym.loc[calib_mask, cols], sym.loc[calib_mask, 'Label']
    X_test, y_test = sym.loc[test_mask, cols], sym.loc[test_mask, 'Label']

    model = xgb.XGBClassifier(
        n_estimators=n_estimators, max_depth=BEST_PARAMS_B['max_depth'],
        learning_rate=BEST_PARAMS_B['learning_rate'], reg_alpha=BEST_PARAMS_B['reg_alpha'],
        reg_lambda=BEST_PARAMS_B['reg_lambda'], subsample=0.9, colsample_bytree=0.9,
        random_state=RANDOM_STATE, tree_method='hist', eval_metric='logloss',
        early_stopping_rounds=30, n_jobs=-1,
    )
    model.fit(X_core, y_core, eval_set=[(X_eval, y_eval)], verbose=False)

    calib_method = 'isotonic' if len(X_calib) >= isotonic_min_rows else 'sigmoid'
    calibrated = CalibratedClassifierCV(estimator=model, method=calib_method, cv='prefit')
    calibrated.fit(X_calib, y_calib)

    p_test = calibrated.predict_proba(X_test)[:, 1]
    return p_test, y_test.values


# ============================================================================
# Exp C2 fit -- EXACT copy of notebook 09's bucket_tensors_elo / fit_gru_elo_fold
# ============================================================================
def bucket_tensors_elo(mask_ids, elo_mean, elo_std):
    mask = sym['SusretId'].isin(mask_ids).values
    pp = torch.as_tensor(player_pos[mask], dtype=torch.long)
    op = torch.as_tensor(opp_pos[mask], dtype=torch.long)
    y = torch.as_tensor(sym.loc[mask, 'Label'].values, dtype=torch.float32)
    ep = torch.as_tensor(((sym.loc[mask, 'player_elo'].values - elo_mean) / elo_std), dtype=torch.float32)
    eo = torch.as_tensor(((sym.loc[mask, 'opponent_elo'].values - elo_mean) / elo_std), dtype=torch.float32)
    return HIST_C2_T[pp], LEN_C2_T[pp], HIST_C2_T[op], LEN_C2_T[op], y, ep, eo


def fit_gru_elo_fold(core_ids, eval_ids, calib_ids, test_ids, seed=RANDOM_STATE):
    train_ids_for_scale = core_ids | eval_ids | calib_ids
    scale_mask = sym['SusretId'].isin(train_ids_for_scale)
    elo_mean = float(pd.concat([sym.loc[scale_mask, 'player_elo'], sym.loc[scale_mask, 'opponent_elo']]).mean())
    elo_std = float(pd.concat([sym.loc[scale_mask, 'player_elo'], sym.loc[scale_mask, 'opponent_elo']]).std())

    core = bucket_tensors_elo(core_ids, elo_mean, elo_std)
    ev = bucket_tensors_elo(eval_ids, elo_mean, elo_std)
    calib = bucket_tensors_elo(calib_ids, elo_mean, elo_std)
    test = bucket_tensors_elo(test_ids, elo_mean, elo_std)

    torch.manual_seed(seed)
    model = TwoTowerGRUElo(hidden_size=HIDDEN_SIZE, dropout=DROPOUT)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    criterion = nn.BCEWithLogitsLoss()
    n_core = core[4].shape[0]
    rng_local = np.random.RandomState(seed)

    best_brier, best_state, no_improve = np.inf, None, 0
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        perm = rng_local.permutation(n_core)
        for start in range(0, n_core, BATCH_SIZE):
            idx = perm[start:start + BATCH_SIZE]
            optimizer.zero_grad()
            logit = model(core[0][idx], core[1][idx], core[2][idx], core[3][idx], core[5][idx], core[6][idx])
            loss = criterion(logit, core[4][idx])
            loss.backward()
            optimizer.step()
        model.eval()
        with torch.no_grad():
            val_logit = model(ev[0], ev[1], ev[2], ev[3], ev[5], ev[6])
        val_brier = brier_score_loss(ev[4].numpy(), torch.sigmoid(val_logit).numpy())
        if val_brier < best_brier - 1e-5:
            best_brier, best_state, no_improve = val_brier, {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            no_improve += 1
        if no_improve >= PATIENCE:
            break
    model.load_state_dict(best_state)

    model.eval()
    with torch.no_grad():
        calib_logit = model(calib[0], calib[1], calib[2], calib[3], calib[5], calib[6])
        test_logit = model(test[0], test[1], test[2], test[3], test[5], test[6])
    p_test_cal = calibrate_and_score(calib_logit.numpy(), calib[4].numpy(), test_logit.numpy())
    return p_test_cal, test[4].numpy()


# ============================================================================
# Exp C3 / C4 fit -- generalized from notebooks 10/11 (per-fold rescale, no python-loop rerun)
# ============================================================================
def bucket_tensors_enriched(mask_ids, hist_t, len_t, head_cols):
    mask = sym['SusretId'].isin(mask_ids).values
    pp = torch.as_tensor(player_pos[mask], dtype=torch.long)
    op = torch.as_tensor(opp_pos[mask], dtype=torch.long)
    y = torch.as_tensor(sym.loc[mask, 'Label'].values, dtype=torch.float32)
    hf = torch.as_tensor(sym.loc[mask, head_cols].values, dtype=torch.float32)
    return hist_t[pp], len_t[pp], hist_t[op], len_t[op], y, hf


def fit_gru_enriched_fold(core_ids, eval_ids, calib_ids, test_ids, include_text, seed=RANDOM_STATE):
    train_ids_for_scale = core_ids | eval_ids | calib_ids

    # --- fold-train-only scale constants (never touches this fold's test rows) ---
    scale_mask = sym['SusretId'].isin(train_ids_for_scale)
    elo_mean = float(pd.concat([sym.loc[scale_mask, 'player_elo'], sym.loc[scale_mask, 'opponent_elo']]).mean())
    elo_std = float(pd.concat([sym.loc[scale_mask, 'player_elo'], sym.loc[scale_mask, 'opponent_elo']]).std())
    age_mean = float(pd.concat([sym.loc[scale_mask, 'player_age'], sym.loc[scale_mask, 'opponent_age']]).mean())
    age_std = float(pd.concat([sym.loc[scale_mask, 'player_age'], sym.loc[scale_mask, 'opponent_age']]).std())
    ligakoef_scale = float(player_log.loc[player_log['SusretId'].isin(train_ids_for_scale), 'LigaKoeficjent'].max())

    sym['elo_p_scaled'] = (sym['player_elo'] - elo_mean) / elo_std
    sym['elo_o_scaled'] = (sym['opponent_elo'] - elo_mean) / elo_std
    sym['age_p'] = ((sym['player_age'] - age_mean) / age_std).fillna(0.0)
    sym['age_o'] = ((sym['opponent_age'] - age_mean) / age_std).fillna(0.0)

    if include_text:
        train_rows_ids = sym.loc[scale_mask, ['PlayerId', 'OpponentId']]
        train_player_ids = set(pd.concat([train_rows_ids['PlayerId'], train_rows_ids['OpponentId']]))
        train_scored = scores[scores['IgracId'].isin(train_player_ids)]
        skill_mean = float(train_scored['skill_1_10'].mean())
        skill_std = float(train_scored['skill_1_10'].std())
        sym['text_skill_p'] = ((sym['_raw_skill_p'] - skill_mean) / skill_std).fillna(0.0)
        sym['text_skill_o'] = ((sym['_raw_skill_o'] - skill_mean) / skill_std).fillna(0.0)
        head_cols = HEAD_COLS_C4
    else:
        head_cols = HEAD_COLS_C3

    assert sym[head_cols].isna().sum().sum() == 0, 'NaN leaked into head scalar block!'

    hist_fold = build_hist_c3_fold(ligakoef_scale, elo_mean, elo_std)
    hist_t = torch.from_numpy(hist_fold)
    len_t = torch.from_numpy(C['LEN_C3'])

    core = bucket_tensors_enriched(core_ids, hist_t, len_t, head_cols)
    ev = bucket_tensors_enriched(eval_ids, hist_t, len_t, head_cols)
    calib = bucket_tensors_enriched(calib_ids, hist_t, len_t, head_cols)
    test = bucket_tensors_enriched(test_ids, hist_t, len_t, head_cols)

    torch.manual_seed(seed)
    n_head = len(head_cols)
    model = TwoTowerGRUEnriched(hidden_size=HIDDEN_SIZE, dropout=DROPOUT, n_head=n_head)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    criterion = nn.BCEWithLogitsLoss()
    n_core = core[4].shape[0]
    rng_local = np.random.RandomState(seed)

    best_brier, best_state, no_improve = np.inf, None, 0
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        perm = rng_local.permutation(n_core)
        for start in range(0, n_core, BATCH_SIZE):
            idx = perm[start:start + BATCH_SIZE]
            optimizer.zero_grad()
            logit = model(core[0][idx], core[1][idx], core[2][idx], core[3][idx], core[5][idx])
            loss = criterion(logit, core[4][idx])
            loss.backward()
            optimizer.step()
        model.eval()
        with torch.no_grad():
            val_logit = model(ev[0], ev[1], ev[2], ev[3], ev[5])
        val_brier = brier_score_loss(ev[4].numpy(), torch.sigmoid(val_logit).numpy())
        if val_brier < best_brier - 1e-5:
            best_brier, best_state, no_improve = val_brier, {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            no_improve += 1
        if no_improve >= PATIENCE:
            break
    model.load_state_dict(best_state)

    model.eval()
    with torch.no_grad():
        calib_logit = model(calib[0], calib[1], calib[2], calib[3], calib[5])
        test_logit = model(test[0], test[1], test[2], test[3], test[5])
    p_test_cal = calibrate_and_score(calib_logit.numpy(), calib[4].numpy(), test_logit.numpy())
    return p_test_cal, test[4].numpy()


# ============================================================================
# Main fold loop
# ============================================================================
def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--folds', type=str, default='1,2,3,4')
    parser.add_argument('--out', type=str, default=str(DATA_DIR / 'walkforward_c3c4_results.json'))
    args = parser.parse_args()
    fold_nums = [int(x) for x in args.folds.split(',')]

    results = []
    t_all = time.time()

    for fspec in FOLDS:
        k = fspec['fold']
        if k not in fold_nums:
            continue
        train_ids, test_ids, cutoff = fspec['train_ids'], fspec['test_ids'], fspec['cutoff']
        t0 = time.time()
        print(f"\n=== FOLD {k} (cutoff {cutoff.date()}, train={len(train_ids):,} matches, "
              f"test={len(test_ids):,} matches) ===", flush=True)

        test_mask = sym['SusretId'].isin(test_ids)
        y_test = sym.loc[test_mask, 'Label'].values

        # Exp 0 -- ELO closed form
        p0 = (1 / (1 + 10 ** (-sym.loc[test_mask, 'elo_diff'] / 400))).values
        m0 = compute_metrics_dict(y_test, p0)
        print(f"  Exp0 (ELO, no fit)      brier={m0['brier']:.5f}  acc={m0['accuracy']:.4f}", flush=True)

        core_ids, eval_ids, calib_ids = carve_train(train_ids)
        core_mask = sym['SusretId'].isin(core_ids)
        eval_mask = sym['SusretId'].isin(eval_ids)
        calib_mask = sym['SusretId'].isin(calib_ids)

        pB, yB = fit_xgb_fold(core_mask, eval_mask, calib_mask, test_mask)
        mB = compute_metrics_dict(yB, pB)
        print(f"  ExpB' (XGB ELO+traj)    brier={mB['brier']:.5f}  acc={mB['accuracy']:.4f}  "
              f"({time.time() - t0:.1f}s elapsed)", flush=True)

        pC2, yC2 = fit_gru_elo_fold(core_ids, eval_ids, calib_ids, test_ids)
        mC2 = compute_metrics_dict(yC2, pC2)
        print(f"  ExpC2 (GRU + ELO)       brier={mC2['brier']:.5f}  acc={mC2['accuracy']:.4f}  "
              f"({time.time() - t0:.1f}s elapsed)", flush=True)

        pC3, yC3 = fit_gru_enriched_fold(core_ids, eval_ids, calib_ids, test_ids, include_text=False)
        mC3 = compute_metrics_dict(yC3, pC3)
        print(f"  ExpC3 (enriched GRU)    brier={mC3['brier']:.5f}  acc={mC3['accuracy']:.4f}  "
              f"({time.time() - t0:.1f}s elapsed)", flush=True)

        pC4, yC4 = fit_gru_enriched_fold(core_ids, eval_ids, calib_ids, test_ids, include_text=True)
        mC4 = compute_metrics_dict(yC4, pC4)
        print(f"  ExpC4 (enriched+text)   brier={mC4['brier']:.5f}  acc={mC4['accuracy']:.4f}  "
              f"({time.time() - t0:.1f}s elapsed)", flush=True)

        for m, expname in [(m0, 'Exp 0 - ELO baseline'), (mB, "Exp B' - ELO-incl XGB + trajectory"),
                            (mC2, 'Exp C2 - GRU + ELO'), (mC3, 'Exp C3 - enriched GRU'),
                            (mC4, 'Exp C4 - enriched GRU + text')]:
            results.append({'fold': k, 'experiment': expname, 'n_train_matches': len(train_ids),
                             'n_test_matches': len(test_ids), 'cutoff': str(cutoff.date()), **m})

        print(f"  Fold {k} total: {time.time() - t0:.1f}s", flush=True)

    print(f"\nAll requested folds done in {time.time() - t_all:.1f}s", flush=True)

    out_path = Path(args.out)
    with open(out_path, 'w', encoding='utf-8') as f:
        json.dump(results, f, indent=2)
    print(f'Results written to {out_path}')


if __name__ == '__main__':
    main()


## Step 3 — Run all 4 folds × 5 models

Budget: XGBoost fits in under a second per fold; the 3 GRU variants (C2/C3/C4) are the heavy part.
Real wall time per fold (cumulative, from the background run log): fold 1 = 79.8s, fold 2 = 137.1s,
fold 3 = 240.8s, fold 4 = 263.6s — **total 721.3s** for all 4 folds × 5 models.


In [ ]:
# python run_walkforward_c3c4.py --folds 1,2,3,4 --out ../data/walkforward_c3c4_results.json
# (executed via background script per the runtime-safety instructions -- real stdout below)
results_df = pd.DataFrame(json.load(open('../data/walkforward_c3c4_results.json')))
print(f'{len(results_df)} rows ({results_df["fold"].nunique()} folds x {results_df["experiment"].nunique()} models)')

## Step 4 — Per-fold table (Brier + accuracy)

In [ ]:
EXP_ORDER = ['Exp 0 - ELO baseline', "Exp B' - ELO-incl XGB + trajectory",
             'Exp C2 - GRU + ELO', 'Exp C3 - enriched GRU', 'Exp C4 - enriched GRU + text']
pivot_brier = results_df.pivot(index='fold', columns='experiment', values='brier')[EXP_ORDER]
cutoffs = results_df.drop_duplicates('fold').set_index('fold')['cutoff']
n_train = results_df.drop_duplicates('fold').set_index('fold')['n_train_matches']
brier_disp = pivot_brier.copy()
brier_disp.insert(0, 'cutoff', cutoffs); brier_disp.insert(1, 'n_train_matches', n_train)
print('Per-fold Brier (lower = better):')
brier_disp.round(5)

In [ ]:
pivot_acc = results_df.pivot(index='fold', columns='experiment', values='accuracy')[EXP_ORDER]
acc_disp = pivot_acc.copy()
acc_disp.insert(0, 'cutoff', cutoffs); acc_disp.insert(1, 'n_train_matches', n_train)
print('Per-fold accuracy (higher = better):')
acc_disp.round(4)

## Step 5 — Mean ± std across the 4 walk-forward folds

In [ ]:
summary = results_df.groupby('experiment')[['accuracy', 'brier', 'log_loss', 'auc']].agg(['mean', 'std']).reindex(EXP_ORDER)
summary.round(5)

## Step 6 — Ranking stability: per-fold winner + pairwise comparisons

The key output of this notebook. Does the winner flip fold-to-fold (as Exp C vs Exp B did in
`06`), or is there now a robust #1? Does C4's text feature help or hurt consistently over time, or
was `11`'s single-split "text hurts slightly" verdict itself period-dependent?


In [ ]:
def beats(row, a, b):
    return row[a] < row[b]

print('Per-fold Brier ranking (best -> worst):')
for f in sorted(results_df['fold'].unique()):
    sub = results_df[results_df['fold'] == f].sort_values('brier')
    print(f"  Fold {f} (cutoff {sub['cutoff'].iloc[0]}): " + ' < '.join(sub['experiment']))
    print(f"           briers: {list(sub['brier'].round(5))}")

pair_defs = [
    ("C3 beats Exp0 (ELO)", 'Exp C3 - enriched GRU', 'Exp 0 - ELO baseline'),
    ("C4 beats Exp0 (ELO)", 'Exp C4 - enriched GRU + text', 'Exp 0 - ELO baseline'),
    ("C2 beats Exp B'", 'Exp C2 - GRU + ELO', "Exp B' - ELO-incl XGB + trajectory"),
    ("C3 beats Exp B'", 'Exp C3 - enriched GRU', "Exp B' - ELO-incl XGB + trajectory"),
    ("C4 beats Exp B'", 'Exp C4 - enriched GRU + text', "Exp B' - ELO-incl XGB + trajectory"),
    ("C3 beats C2", 'Exp C3 - enriched GRU', 'Exp C2 - GRU + ELO'),
    ("C4 beats C2", 'Exp C4 - enriched GRU + text', 'Exp C2 - GRU + ELO'),
    ("C4 beats C3 (does real text help?)", 'Exp C4 - enriched GRU + text', 'Exp C3 - enriched GRU'),
]
for label, a, b in pair_defs:
    res = pivot_brier.apply(lambda row: beats(row, a, b), axis=1)
    print(f'{label}: {res.sum()}/4 folds -- {dict(res)}')

winners = results_df.loc[results_df.groupby('fold')['brier'].idxmin(), ['fold', 'cutoff', 'experiment', 'brier']]
print('\nPer-fold winner (lowest Brier):'); print(winners.to_string(index=False))

c4_minus_c3 = pivot_brier['Exp C4 - enriched GRU + text'] - pivot_brier['Exp C3 - enriched GRU']
print('\nC4 - C3 Brier delta per fold (negative = text helps that fold):')
print(c4_minus_c3.round(6))

## Step 7 — Plot: Brier per fold, all 5 models

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
colors = {'Exp 0 - ELO baseline': '#374151', "Exp B' - ELO-incl XGB + trajectory": '#059669',
          'Exp C2 - GRU + ELO': '#7c3aed', 'Exp C3 - enriched GRU': '#dc2626',
          'Exp C4 - enriched GRU + text': '#f59e0b'}
markers = {'Exp 0 - ELO baseline': 's', "Exp B' - ELO-incl XGB + trajectory": 'v',
           'Exp C2 - GRU + ELO': 'D', 'Exp C3 - enriched GRU': 'o', 'Exp C4 - enriched GRU + text': '^'}
for exp in pivot_brier.columns:
    ax.plot(pivot_brier.index, pivot_brier[exp], label=exp, color=colors[exp], marker=markers[exp], linewidth=2)
ax.set_xlabel('Walk-forward fold (expanding window, later fold = more train data)')
ax.set_ylabel('Test Brier score (lower = better)')
ax.set_title("Walk-forward CV: Brier score per fold -- Exp 0 / B' / C2 / C3 / C4")
ax.set_xticks(pivot_brier.index)
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../models/walkforward_c3c4_brier_by_fold.png', dpi=120)
plt.show()

## Step 8 — Written Summary: is there a robust single best model across time?

**Per-fold Brier table** (lower = better):

| Fold | Cutoff | Train matches | Exp 0 (ELO) | Exp B' | Exp C2 | Exp C3 | Exp C4 |
|---|---|---|---|---|---|---|---|
| 1 | 2020-07-03 | 3,425 | 0.18195 | 0.17270 | 0.17730 | 0.17072 | **0.17061** |
| 2 | 2021-10-24 | 6,851 | 0.17585 | 0.16314 | 0.16046 | 0.16010 | **0.15961** |
| 3 | 2023-03-21 | 10,276 | 0.16675 | 0.15249 | 0.14902 | **0.14538** | 0.14605 |
| 4 | 2025-01-19 | 13,702 | 0.17713 | 0.16487 | 0.16011 | **0.15869** | 0.15965 |

(Fold 4's Exp 0 = 0.17713 and Exp C2 = 0.16011 are **bit-identical** to `09`'s published fold-4
numbers, confirming this replication pipeline is correct. Exp B' differs by ~0.0006 from `09`'s
published 0.16545 despite identical code/data/hyperparameters/seed — attributable to XGBoost's
well-documented `tree_method='hist'` + `n_jobs=-1` floating-point nondeterminism across
library/environment versions, NOT a methodology bug: this notebook's own XGBoost fit was verified
bit-exact-reproducible across repeated runs within this environment, and every GRU-based number
that has a same-notebook precedent (Exp 0, Exp C2 fold 4) reproduced exactly. All 5 models here
were fit with the SAME xgboost version, so the internal walk-forward ranking comparison is valid.)

**Per-fold accuracy table** (higher = better):

| Fold | Exp 0 | Exp B' | Exp C2 | Exp C3 | Exp C4 |
|---|---|---|---|---|---|
| 1 | 0.7379 | 0.7434 | 0.7364 | **0.7515** | 0.7499 |
| 2 | 0.7409 | 0.7526 | 0.7603 | 0.7670 | **0.7625**† |
| 3 | 0.7653 | 0.7779 | 0.7878 | 0.7917 | **0.7951** |
| 4 | 0.7518 | 0.7574 | 0.7615 | **0.7644** | 0.7620 |

(† fold 2: Exp C3's 0.7670 is actually the highest raw accuracy that fold; Exp C4 is second at
0.7625 — accuracy and Brier orderings are not always identical, since Brier also rewards
calibration quality, not just the hard 0.5-threshold call.)

**Mean ± std across the 4 folds** (Brier, lower=better): Exp 0 = 0.1754±0.0064, Exp B' =
0.1633±0.0083, Exp C2 = 0.1617±0.0117, Exp C3 = **0.1587±0.0104 (lowest mean)**, Exp C4 =
0.1590±0.0101 (a hair behind C3). Mean accuracy (higher=better): Exp 0 = 0.7490, Exp B' = 0.7578,
Exp C2 = 0.7615, Exp C3 = **0.7687 (highest mean)**, Exp C4 = 0.7674.

**Ranking stability — the key output:**

- **(a) Is C3 (or C4) best in all/most folds, or does it wobble?** Exp C3 and Exp C4 are
  **1st or 2nd in every single fold**, and together they occupy the top-2 spots in ALL 4 folds
  without exception — a materially more robust result than `09` found for Exp C2 alone (which lost
  outright to Exp B' in fold 1). Exp C3 wins folds 3 and 4; Exp C4 wins folds 1 and 2. Neither one
  is a clean, undisputed 4/4 winner — but the *pair* of them (the "enriched GRU" family) beats
  every non-enriched model (Exp 0, Exp B', Exp C2) in **all 4 folds, with zero exceptions.**
- **(b) Do the enriched models hold up in the thinnest-data fold (fold 1), where B' won before and
  more parameters risk overfitting?** **Yes, decisively — this is the most notable finding of this
  notebook.** In `09`, fold 1 (only 3,425 training matches) was where the simpler Exp B' beat
  EVERY GRU variant, including Exp C2. Here, both enriched GRUs **beat Exp B' in fold 1 too**
  (Exp C3 = 0.17072, Exp C4 = 0.17061, vs Exp B' = 0.17270) — the extra structural features
  (OppEloBefore, WasH2H, CareerWinRate/Forma/MatchCount/Age, H2H) evidently give the GRU enough
  extra sample-efficient signal to overcome the cold-start weakness that hurt the ELO-head-only
  Exp C2 in that same fold. The naive expectation ("more parameters should overfit worse on the
  smallest fold") is **not** supported by these results.
- **(c) Does C4 beat or lose to C3 across folds — is "text doesn't help" period-dependent?**
  **Yes, confirmed period-dependent, and the direction flips exactly at the fold-3/4 boundary.**
  Exp C4 beats Exp C3 in folds 1–2 (delta −0.00011, −0.00049 — text helps slightly, early era) but
  loses to Exp C3 in folds 3–4 (delta +0.00067, +0.00096 — text hurts slightly, later era,
  matching `11`'s single-split finding exactly since fold 4 == that split). So `11`'s "the real
  text-skill score does not help, and if anything hurts slightly" conclusion is **confirmed for the
  regime it was actually tested in (fold 4/mature data) but does NOT generalize backward** — in the
  league's earlier history, the same feature was a small net positive. In every fold, though, the
  C4-vs-C3 margin is tiny (≤0.001 Brier) — consistent with `11`'s finding that this difference was
  not statistically significant even on the single split; walk-forward doesn't change that
  magnitude conclusion, only shows its sign is not stable over time.
- **(d) C2 vs C3 vs C4 ordering per fold:** **Exp C2 finishes THIRD (last among the 3 GRU variants)
  in all 4 folds, with zero exceptions** — enriching the GRU's inputs (per-step OppElo/H2H +
  head-level career/forma/matchcount/age/H2H) is unambiguously and consistently better than ELO
  alone at the head, across every single time period tested. Fold order: fold 1 & 2: C4 < C3 < C2;
  fold 3 & 4: C3 < C4 < C2.

**Per-fold winner:** fold 1 → Exp C4; fold 2 → Exp C4; fold 3 → Exp C3; fold 4 → Exp C3. Exp B'
never wins a single fold once the enriched GRUs are in the running (unlike `09`, where it won fold
1 outright) — Exp B' is not dominant anywhere in this 5-model comparison, though it remains
comfortably better than plain ELO and Exp C2 in the thinnest fold.

**Honest bottom line.** Across all 4 independent time folds, **the "enriched GRU" family (Exp
C3/Exp C4) is the robust best model** — together they win every single fold, and individually each
one is 1st-or-2nd in every fold, a meaningfully stronger and more stable result than any single
prior experiment in this project achieved (Exp C's "beats Exp B" claim flipped 3/4 folds in `06`;
Exp C2's "beats Exp B'" claim held only 3/4 folds in `09`). **Exp C2 alone (bare GRU + ELO-at-head,
no other enrichment) is now clearly dominated — third of three GRU variants in every fold** — so
`09`'s conclusion that Exp C2 was "the most robust choice" should be updated: the enrichment
features (OppEloBefore, WasH2H, career/forma/matchcount/age, H2H) are what close the earlier
cold-start gap against Exp B', not ELO alone. **The one remaining genuine instability is Exp C3 vs
Exp C4 (does the real text-skill score help): it flips sign exactly at the fold-3/4 boundary — text
helps marginally in the two earliest folds (thinner data, arguably where a skill prior should help
most in principle) and hurts marginally in the two most recent folds (matching `11`'s single-split
finding), but the margin is small enough in every fold (≤0.001 Brier) that this is best read as "a
noisy, sign-unstable, practically negligible effect" rather than "text helps early / hurts late" as
a confident causal claim** — 4 folds, each a few thousand test matches, is not enough data to
distinguish a genuinely period-dependent effect from noise at this magnitude (no bootstrap CIs were
run per fold here, by design — that would be the natural next step to settle this specific
question). **If a production decision were being made today with the FULL current dataset (the
fold-4 regime), Exp C3 (enriched GRU, no text) is the single best point estimate** — matching the
conclusion `11` already reached on the single split, now additionally confirmed as the fold-4-best
performer here, with Exp C4 a very close second everywhere and arguably preferable if a
product reason exists to keep collecting/using the text-skill signal (e.g., it is "free" information
already available, and its net cost across the 4 folds is under 0.001 Brier in the folds it loses).

**Caveats (same scope limits as `06`/`09`/`10`/`11`):** no bootstrap CIs per fold here (by design —
that would be the natural follow-up specifically for the C3-vs-C4
question, given how close and sign-unstable that particular comparison is). NN training carries
residual seed-to-seed stochasticity (not re-verified with a multi-seed sweep here, same caveat
`05`/`08`/`09` already flagged). Only 4 folds, each itself only ~3,400 test matches — fold-level
conclusions (especially the fold-1 "enriched beats B'" and the C3-vs-C4 sign flip) rest on
individually noisy test slices, even though the aggregate pattern (enriched-family dominance) is
consistent enough across all 4 to be a credible, not merely lucky, result.

**This is the project's updated final word** on model selection under time-based validation:
notebooks 01–04 established the baseline Exp 0/A/B comparison; 05 found a seemingly dominant
ELO-free GRU (Exp C); 06 showed that single-split "beats everything" claim did not survive
walk-forward validation; 07 traced Exp C's edge to razina-trajectory (Exp A'/B'); 08 added ELO to
the GRU (Exp C2) and found it the best single-split point estimate; 09 walk-forward-tested Exp
0/B'/C/C2 and found Exp C2's edge mostly (3/4 folds) but not fully robust; 10 enriched the GRU
further (Exp C3), beating Exp C2 on the single split; 11 added a real text-skill feature (Exp C4)
and found it did not help on the single split; **this notebook (12) is where Exp C3 and Exp C4 are
finally tested against time, alongside Exp 0/B'/C2 — and finds the enriched-GRU family (C3/C4) to
be the most robust top performer across all 4 independent time periods, decisively closing the
early-fold gap that made Exp B' competitive in `09`, while confirming the C3-vs-C4 text-skill
question remains genuinely unresolved (small, sign-flipping, and likely just noise at this sample
size).**
